In [ ]:
import warnings
warnings.filterwarnings("ignore")
import sys
sys.path.append("..")

In [ ]:
import torch
from torch.utils.data import DataLoader
import torch.fft

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import matplotlib.colors as mcolors

import numpy as np

import time

import scipy.sparse as sp
import scipy.sparse.linalg as spla

import soap
from architectures import PINO2D
from generators import RibWaveguideDataset
from utils import full_vectorial_fd

In [ ]:
ds = RibWaveguideDataset(n_dataset=1, device='cpu')
dl = DataLoader(ds, batch_size=1, shuffle=False)
for batch in dl:
    print(batch[0].shape, batch[1].shape)
    plt.imshow(batch[0][0, 2, :, :].flip((0,)))

In [ ]:
def d1_1d(n, d):
    """1st derivative central difference matrix"""
    off_diag_p = np.ones(n - 1)
    off_diag_m = -np.ones(n - 1)
    return sp.diags([off_diag_m, np.zeros(n), off_diag_p], [-1, 0, 1]) / (2 * d)

def d2_1d(n, d):
    """2nd derivative central difference matrix"""
    main_diag = -2 * np.ones(n)
    off_diag = np.ones(n - 1)
    return sp.diags([off_diag, main_diag, off_diag], [-1, 0, 1]) / d**2

def is_symmetric(A, tol=1e-10):
    if not sp.issparse(A):
        A = sp.csr_matrix(A)  # convert to sparse if not already
    diff = A - A.T
    if diff.nnz == 0:
        return True
    else:
        return np.allclose(diff.data, 0, atol=tol)

def full_vectorial_fd(xy, eps_map, dx, dy, k0, num_modes=6):
    """
    Full-vectorial eigenmode solver (2D FD) based on the equations:
    ∂²Ex/∂x² + ∂²Ex/∂y² + ∂/∂x[(1/ε)(∂ε/∂x)Ex] + ∂/∂x[(1/ε)(∂ε/∂y)Ey] + (β² - k₀²ε)Ex = 0
    ∂²Ey/∂x² + ∂²Ey/∂y² + ∂/∂y[(1/ε)(∂ε/∂y)Ey] + ∂/∂y[(1/ε)(∂ε/∂x)Ex] + (β² - k₀²ε)Ey = 0

    Args:
        xy: (Ny, Nx, 2) torch or np array with coordinates
        eps_map: (Ny, Nx) tensor or ndarray, epsilon = n²
        dx, dy: step size
        k0: 2*pi / lambda
        num_modes: number of modes to compute
    Returns:
        neff: (num_modes,) array of effective indices
        Ex_all: (num_modes, Ny, Nx) array of Ex field components
        Ey_all: (num_modes, Ny, Nx) array of Ey field components
        eps_map: (Ny, Nx) array of permittivity
    """

    # Convert tensors to numpy arrays if needed
    if isinstance(xy, torch.Tensor):
        xy = xy.detach().cpu().numpy()
    if isinstance(eps_map, torch.Tensor):
        eps_map = eps_map.detach().cpu().numpy()

    Ny, Nx = eps_map.shape
    N = Nx * Ny

    # Flatten the permittivity map
    eps_flat = eps_map.flatten()
    eps_diag = sp.diags(eps_flat)

    # Create identity matrices
    Ix = sp.eye(Nx)
    Iy = sp.eye(Ny)

    # Second derivative operators (for Laplacian terms ∂²E/∂x² + ∂²E/∂y²)
    D2x = d2_1d(Nx, dx)
    D2y = d2_1d(Ny, dy)

    # Laplacian operator
    Lx_op = sp.kron(Iy, D2x)
    Ly_op = sp.kron(D2y, Ix)
    Laplacian = Lx_op + Ly_op

    # First derivative operators
    Dx_1d = d1_1d(Nx, dx)
    Dy_1d = d1_1d(Ny, dy)
    Dx = sp.kron(Iy, Dx_1d)
    Dy = sp.kron(Dy_1d, Ix)

    # Compute spatial derivatives of permittivity: ∂ε/∂x and ∂ε/∂y
    eps_x = Dx @ eps_flat
    eps_y = Dy @ eps_flat

    # Create diagonal matrices for calculations
    Eps_inv = sp.diags(1.0 / eps_flat)
    Eps_x = sp.diags(eps_x)
    Eps_y = sp.diags(eps_y)

    # ---- Constructing the operator terms for the vectorial wave equation ----

    # Term: ∂/∂x[(1/ε)(∂ε/∂x)Ex] for Ex equation
    T_xx = Dx @ (Eps_inv @ Eps_x)

    # Term: ∂/∂x[(1/ε)(∂ε/∂y)Ey] for Ex equation
    T_xy = Dx @ (Eps_inv @ Eps_y)

    # Term: ∂/∂y[(1/ε)(∂ε/∂y)Ey] for Ey equation
    T_yy = Dy @ (Eps_inv @ Eps_y)

    # Term: ∂/∂y[(1/ε)(∂ε/∂x)Ex] for Ey equation
    T_yx = Dy @ (Eps_inv @ Eps_x)

    # Constructing the diagonal blocks (self-coupling terms)
    A11 = Laplacian + T_xx + k0**2 * eps_diag  # Operator for Ex
    A22 = Laplacian + T_yy + k0**2 * eps_diag  # Operator for Ey

    # Constructing the off-diagonal blocks (cross-coupling terms)
    A12 = T_xy  # Coupling from Ey to Ex
    A21 = T_yx  # Coupling from Ex to Ey

    # Assemble the full system matrix
    top = sp.hstack([A11, A12])
    bottom = sp.hstack([A21, A22])
    A = sp.vstack([top, bottom]).tocsc()

    # Target eigenvalue - use sigma close to the expected eigenvalues
    # β² will be our eigenvalues, so set sigma near k0²ε
    sigma = (k0**2 * np.max(eps_flat)) - 0.1

    # Solve the eigenvalue problem: A*[Ex; Ey] = β²*[Ex; Ey]
    try:
        eigvals, eigvecs = spla.eigs(A, k=num_modes, sigma=sigma, which='LM')
    except Exception as e:
        print(f"Eigenvalue computation failed: {e}")
        # Fallback to a more robust but slower method
        eigvals, eigvecs = spla.eigs(A, k=num_modes, which='LM')

    # Sort eigenvalues in descending order
    idx = np.argsort(eigvals.real)[::-1]
    beta2 = eigvals.real[idx]
    neff = np.sqrt(beta2) / k0  # Convert β to effective index

    # Extract all modes
    Ex_all = np.zeros((num_modes, Ny, Nx), dtype=complex)
    Ey_all = np.zeros((num_modes, Ny, Nx), dtype=complex)

    for i, idx_i in enumerate(idx[:num_modes]):
        E_mode = eigvecs[:, idx_i]
        Ex_all[i] = E_mode[:N].reshape(Ny, Nx)
        Ey_all[i] = E_mode[N:].reshape(Ny, Nx)

    return neff[:num_modes], Ex_all, Ey_all, eps_map

In [ ]:
def second_derivative_x_centered(f, xy):
    """
    Computes second derivative ∂²f/∂x² using non-uniform grid finite difference.

    Args:
        f:  (B, 1, Nx, Ny) tensor — scalar field
        xy: (B, 2, Nx, Ny) tensor — coordinates of each grid point

    Returns:
        d2fdx2: (B, 1, Nx, Ny) — second derivative in x
    """
    d2fdx2 = torch.zeros_like(f)

    x = xy[:, 0:1, :, :]  # x-coordinate

    e = x[:, :, 2:, :] - x[:, :, 1:-1, :]  # forward dx
    w = x[:, :, 1:-1, :] - x[:, :, :-2, :] # backward dx

    d2fdx2[:, :, 1:-1, :] = (2 / (e * (e + w))) * f[:, :, 2:, :] \
                          + (2 / (w * (e + w))) * f[:, :, :-2, :] \
                          - (2 / (e * w)) * f[:, :, 1:-1, :]

    d2fdx2[:, :, 0, :] = d2fdx2[:, :, 1, :]
    d2fdx2[:, :, -1, :] = d2fdx2[:, :, -2, :]

    return d2fdx2


def second_derivative_y_centered(f, xy):
    """
    Computes second derivative ∂²f/∂y² using non-uniform grid finite difference.

    Args:
        f:  (B, 1, Nx, Ny) tensor — scalar field
        xy: (B, 2, Nx, Ny) tensor — coordinates of each grid point

    Returns:
        d2fdy2: (B, 1, Nx, Ny) — second derivative in y
    """
    d2fdy2 = torch.zeros_like(f)

    y = xy[:, 1:2, :, :]  # y-coordinate

    s = y[:, :, :, 2:] - y[:, :, :, 1:-1]  # forward dy
    n = y[:, :, :, 1:-1] - y[:, :, :, :-2] # backward dy

    d2fdy2[:, :, :, 1:-1] = (2 / (s * (n + s))) * f[:, :, :, 2:] \
                          + (2 / (n * (n + s))) * f[:, :, :, :-2] \
                          - (2 / (n * s)) * f[:, :, :, 1:-1]

    d2fdy2[:, :, :, 0] = d2fdy2[:, :, :, 1]
    d2fdy2[:, :, :, -1] = d2fdy2[:, :, :, -2]

    return d2fdy2

def second_TE_derivative_x_centered(f, xy, r):
    """
    f: (B, 1, Nx, Ny) tensor
    dx: grid spacing (scalar)
    Returns: second derivative ∂/∂x(1/r ∂r/∂x * E) with same shape
    """
    d_term = torch.zeros_like(f)

    x = xy[:, 0:1, :, :]  # x-coordinate

    e = x[:, :, 2:, :] - x[:, :, 1:-1, :]  # forward dx
    w = x[:, :, 1:-1, :] - x[:, :, :-2, :] # backward dx

    term1 = (1/e)*((r[:, :, 2:, :] - r[:, :, 1:-1, :])/(r[:, :, 2:, :] + r[:, :, 1:-1, :]))*(f[:, :, 2:, :] + f[:, :, 1:-1, :])
    term2 = (1/w)*((r[:, :, 1:-1, :] - r[:, :, :-2, :])/(r[:, :, 1:-1, :] + r[:, :, :-2, :]))*(f[:, :, 1:-1, :] + f[:, :, :-2, :])

    d_term[:, :, 1:-1, :] = (2/(e+w)) * (term1 - term2)

    d_term[:, :, 0, :] = d_term[:, :, 1, :]     # replicate boundary
    d_term[:, :, -1, :] = d_term[:, :, -2, :]

    return d_term

def second_TM_derivative_y_centered(f, xy, r):
    """
    f: (B, 1, Nx, Ny) tensor
    dx: grid spacing (scalar)
    Returns: second derivative ∂/∂y(1/r ∂r/∂y * E) with same shape
    """
    d_term = torch.zeros_like(f)

    y = xy[:, 1:2, :, :]  # y-coordinate

    s = y[:, :, :, 2:] - y[:, :, :, 1:-1]  # forward dy
    n = y[:, :, :, 1:-1] - y[:, :, :, :-2] # backward dy

    term1 = (1/s)*((r[:, :, :, 2:] - r[:, :, :, 1:-1])/(r[:, :, :, 2:] + r[:, :, :, 1:-1]))*(f[:, :, :, 2:] + f[:, :, :, 1:-1])
    term2 = (1/n)*((r[:, :, :, 1:-1] - r[:, :, :, :-2])/(r[:, :, :, 1:-1] + r[:, :, :, :-2]))*(f[:, :, :, 1:-1] + f[:, :, :, :-2])

    d_term[:, :, :, 1:-1] = (2/(s+n)) * (term1 - term2)

    d_term[:, :, :, 0] = d_term[:, :, :, 1]     # replicate boundary
    d_term[:, :, :, -1] = d_term[:, :, :, -2]

    return d_term


def mixed_TE_TM_derivative_xy_nonuniform(f, xy, r):
    """
    Computes ∂/∂x(1/r ∂r/∂y * E) using centered finite differences on a nonuniform grid.

    Args:
        f:  (B, 1, Nx, Ny)
        xy: (B, 2, Nx, Ny)
        r:  (B, 1, Nx, Ny)

    Returns:
        Tensor of shape (B, 1, Nx, Ny)
    """
    B, C, Nx, Ny = f.shape
    x = xy[:, 0:1, :, :]  # (B, 1, Nx, Ny)
    y = xy[:, 1:2, :, :]  # (B, 1, Nx, Ny)

    # Step 1: ∂r/∂y with centered differences
    dy_n = y[:, :, :, 2:] - y[:, :, :, 1:-1]
    dy_s = y[:, :, :, 1:-1] - y[:, :, :, :-2]
    dy = dy_n + dy_s

    dr_dy = (r[:, :, :, 2:] - r[:, :, :, :-2]) / dy        # (B, 1, Nx, Ny-2)
    r_avg = (r[:, :, :, 2:] + r[:, :, :, :-2]) / 2
    E_avg = (f[:, :, :, 2:] + f[:, :, :, :-2]) / 2

    inner = (1.0 / r_avg) * dr_dy * E_avg                  # (B, 1, Nx, Ny-2)

    # Step 2: ∂/∂x of the inner term
    dx_e = x[:, :, 2:, 1:-1] - x[:, :, 1:-1, 1:-1]
    dx_w = x[:, :, 1:-1, 1:-1] - x[:, :, :-2, 1:-1]
    dx = dx_e + dx_w

    inner_central_x = (inner[:, :, 2:, :] - inner[:, :, :-2, :]) / dx  # (B, 1, Nx-2, Ny-2)

    # Step 3: Insert into result tensor
    result = torch.zeros_like(f)
    result[:, :, 1:-1, 1:-1] = inner_central_x

    # Replicate boundary values
    result[:, :, 0, :] = result[:, :, 1, :]
    result[:, :, -1, :] = result[:, :, -2, :]
    result[:, :, :, 0] = result[:, :, :, 1]
    result[:, :, :, -1] = result[:, :, :, -2]

    return result
    
def mixed_TM_TE_derivative_yx_nonuniform(f, xy, r):
    """
    Computes ∂/∂y(1/r ∂r/∂x * E) using centered finite differences on a nonuniform grid.

    Args:
        f:  (B, 1, Nx, Ny)
        xy: (B, 2, Nx, Ny)
        r:  (B, 1, Nx, Ny)

    Returns:
        Tensor of shape (B, 1, Nx, Ny)
    """
    B, C, Nx, Ny = f.shape
    x = xy[:, 0:1, :, :]  # (B, 1, Nx, Ny)
    y = xy[:, 1:2, :, :]  # (B, 1, Nx, Ny)

    # Step 1: ∂r/∂x with centered differences
    dx_e = x[:, :, 2:, :] - x[:, :, 1:-1, :]
    dx_w = x[:, :, 1:-1, :] - x[:, :, :-2, :]
    dx = dx_e + dx_w

    dr_dx = (r[:, :, 2:, :] - r[:, :, :-2, :]) / dx        # (B, 1, Nx-2, Ny)
    r_avg = (r[:, :, 2:, :] + r[:, :, :-2, :]) / 2
    E_avg = (f[:, :, 2:, :] + f[:, :, :-2, :]) / 2

    inner = (1.0 / r_avg) * dr_dx * E_avg                  # (B, 1, Nx-2, Ny)

    # Step 2: ∂/∂y of the inner term using y spacing
    dy_n = y[:, :, 1:-1, 2:] - y[:, :, 1:-1, 1:-1]  # fixed: use y not x
    dy_s = y[:, :, 1:-1, 1:-1] - y[:, :, 1:-1, :-2]
    dy = dy_n + dy_s

    inner_central_y = (inner[:, :, :, 2:] - inner[:, :, :, :-2]) / dy  # (B, 1, Nx-2, Ny-2)

    # Step 3: Insert into result tensor
    result = torch.zeros_like(f)
    result[:, :, 1:-1, 1:-1] = inner_central_y

    # Replicate boundary values
    result[:, :, 0, :] = result[:, :, 1, :]
    result[:, :, -1, :] = result[:, :, -2, :]
    result[:, :, :, 0] = result[:, :, :, 1]
    result[:, :, :, -1] = result[:, :, :, -2]

    return result

In [ ]:
def helmholtz_residual_loss(
    x, E, ri, features, beta
):
    """
    Computes full-vectorial Helmholtz residual loss for E_x and E_y components.

    Args:
        E: (B, 2, Nx, Ny)
        ri: (B, 1, Nx, Ny)
        x:  (B, 2, Nx, Ny)
        features: (B, F)
        beta: (B,) — propagation constant
    Returns:
        scalar loss, beta regularization, beta², and n_max² * k0²
    """
    B = E.shape[0]
    n2 = ri
    k0 = features[:, [0]]

    n2_max = features[:, [-2]]  # (B, 1)
    n2_min = features[:, [-3]]  # (B, 1)
    beta_sq = (n2_max - beta * (n2_max - n2_min)) * k0**2  # (B, 1)
    beta_sq = beta_sq.view(B, 1, 1, 1)  # (B, 1, 1, 1)

    # Get E_x and E_y once
    Ex = E[:, 0:1, :, :]
    Ey = E[:, 1:2, :, :]

    # Derivatives
    d2Ex_xx = second_derivative_x_centered(Ex, x)
    d2Ex_yy = second_derivative_y_centered(Ex, x)
    dxxphi = second_TE_derivative_x_centered(Ex, x, ri)
    dyxphi = mixed_TM_TE_derivative_yx_nonuniform(Ex, x, ri)

    d2Ey_xx = second_derivative_x_centered(Ey, x)
    d2Ey_yy = second_derivative_y_centered(Ey, x)
    dxyphi = mixed_TE_TM_derivative_xy_nonuniform(Ey, x, ri)
    dyyphi = second_TM_derivative_y_centered(Ey, x, ri)

    # Field equation residuals
    common_term_x = d2Ex_xx + d2Ex_yy + dxxphi + dxyphi
    common_term_y = d2Ey_yy + d2Ey_xx + dyyphi + dyxphi

    phase_term = k0.view(B, 1, 1, 1)**2 * n2 - beta_sq  # broadcast to (B, 1, Nx, Ny)

    residual1 = (common_term_x + phase_term * Ex).pow(2)
    residual2 = (common_term_y + phase_term * Ey).pow(2)

    # Final loss
    loss = residual1.mean() + residual2.mean()
    beta_loss = ((beta-1e-2)**2).mean() # a small bias to prevent premature plateauing

    return loss, beta_loss, beta_sq, n2_max * k0**2

def boundary_condition_loss(E):
    """
    E: (B, 1, Nx, Ny)
    Returns MSE loss enforcing Dirichlet (zero) boundary on all 4 edges.
    """
    left   = E[:, :, 0, :]     # (B, 1, Ny)
    right  = E[:, :, -1, :]    # (B, 1, Ny)
    top    = E[:, :, :, 0]     # (B, 1, Nx)
    bottom = E[:, :, :, -1]    # (B, 1, Nx)

    return (left**2 + right**2).mean() + (top**2 + bottom**2).mean()

In [ ]:
def visualize(xyr, E, modes_x, modes_y):
    plt.figure(figsize=(8, 4))
    plt.subplot(1, 2, 1)
    plt.contour(xyr[0, 0, :, :].cpu(), xyr[0, 1, :, :].cpu(), E[0, 0].cpu().detach(), cmap='jet')
    plt.colorbar()
    plt.contour(xyr[0, 0, :, :].cpu(), xyr[0, 1, :, :].cpu(), modes_x[0], linestyles='dashed', colors='black')
    plt.subplot(1, 2, 2)
    plt.contour(xyr[0, 0, :, :].cpu(), xyr[0, 1, :, :].cpu(), E[0, 1].cpu().detach(), cmap='jet')
    plt.colorbar()
    plt.contour(xyr[0, 0, :, :].cpu(), xyr[0, 1, :, :].cpu(), modes_y[0], linestyles='dashed', colors='black')
    plt.show()
    plt.close()

    plt.figure(figsize=(8, 4))
    plt.subplot(1, 2, 1)
    plt.contour(xyr[0, 0, :, :].cpu(), xyr[0, 1, :, :].cpu(), E[0, 0].cpu().detach(), cmap='jet')
    plt.colorbar()
    plt.contour(xyr[0, 0, :, :].cpu(), xyr[0, 1, :, :].cpu(), modes_x[1], linestyles='dashed', colors='black')
    plt.subplot(1, 2, 2)
    plt.contour(xyr[0, 0, :, :].cpu(), xyr[0, 1, :, :].cpu(), E[0, 1].cpu().detach(), cmap='jet')
    plt.colorbar()
    plt.contour(xyr[0, 0, :, :].cpu(), xyr[0, 1, :, :].cpu(), modes_y[1], linestyles='dashed', colors='black')
    plt.show()
    plt.close()

def train(loader, val_loader, model, optimizer, scheduler, epochs, load, save):
    xyr_val, features_val = next(iter(val_loader))
    batch_indices = torch.arange(len(loader.dataset))
    dxdy = dataset.dx * dataset.dy
    
    train_loss_history = []
    val_loss_history = []
    betas_avg = []

    epoch = 0
    checkpoint_no = 0
    if load: 
        checkpoint_no = np.load('last_checkpoint.npy').item()
        checkpoint = torch.load(f"checkpoint_{checkpoint_no}.pth", weights_only=False)
        
        epoch = checkpoint['epoch']
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        
        train_loss_history = checkpoint["train_loss_history"]
        val_loss_history = checkpoint["val_loss_history"]
        betas_avg = checkpoint["betas_avg"]
        
        xyr_val, features_val = checkpoint['xyr_val'], checkpoint['features_val']
        
        print('loaded latest parameteres')

    print('there are', len(xyr_val), "validation_examples")

    for iteration_no in range(epoch, epochs):
        for (i, (xyr, features)) in enumerate(loader):
            selected_components = features[:, -1].to(torch.int)
            E_base, beta = model(xyr)                 # (B, 2, N)

            E_out = torch.zeros_like(E_base)
            E_out[batch_indices, selected_components] = E_base[:, 0]
            E_out[batch_indices, 1 - selected_components] = E_base[:, 1]
            E = E_out / torch.sqrt(dxdy * E_out.pow(2).sum(dim=(1, 2, 3), keepdim=True))

            # weight =  1e2 * np.exp(-iteration_no/500)
            weight =  1e2 * max(0, 1-iteration_no/1000)

            # Losses
            bc_loss = boundary_condition_loss(E)
            helm_loss, beta_loss, beta, n2_max = helmholtz_residual_loss(xyr[:, :2], E, xyr[:, 2:3], features, beta=beta)
            loss = helm_loss + bc_loss * 1e1 + beta_loss  * weight

            train_loss_history.append(helm_loss.item() + bc_loss.item())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        scheduler.step()


        if iteration_no % 250 == 0:
            print(f"Epoch {iteration_no}:")
            print(f"  β² ≈ {beta.squeeze().detach().cpu().numpy()}")
            print(f"  β² ≈ {n2_max.squeeze().detach().cpu().numpy()}")

            print(f"Helmholtz Loss: {helm_loss.item():.6e}  BC Loss: {bc_loss.item():.6e} Beta Loss {beta_loss.item():.6e} Total Loss: {loss.item():.6e}")


            if iteration_no % 1000 == 0:
                print('Predicting a', 'TM' if features[0, -1] else 'TE', 'dominated field')
                eig_vals, modes_y, modes_x, _ = full_vectorial_fd(
                    xy=xyr[0, :2].permute(1, 2, 0),
                    eps_map=xyr[0, 2],
                    dx=dataset.dx,
                    dy=dataset.dy,
                    k0=features[0, 0].item(),
                    num_modes=2
                )

                visualize(xyr, E, modes_x, modes_y)
            step = 64
            betas_avg.append(0)
            for start_index in range(0, len(xyr_val), step):
                feature_val_batch = features_val[start_index:start_index+step]
                xyr_val_batch = xyr_val[start_index:start_index+step]

                batch_indicees_val = torch.arange(step, device=xyr_val.device)
                selected_components_val = feature_val_batch[:, -1].to(torch.int)

                with torch.no_grad():
                    E_base, eta = model(xyr_val_batch)
                    E_base = E_base.to(xyr_val.device)
                    eta= eta.to(xyr_val.device)
                    E_out = torch.zeros_like(E_base)
                    E_out[batch_indicees_val, selected_components_val] = E_base[:, 0]
                    E_out[batch_indicees_val, 1 - selected_components_val] = E_base[:, 1]
                    E = E_out / torch.sqrt(dxdy * E_out.pow(2).sum(dim=(1, 2, 3), keepdim=True))
    
                    helm_loss, beta_loss, beta, n2_max = helmholtz_residual_loss(xyr_val_batch[:, :2], E, xyr_val_batch[:, 2:3], feature_val_batch, beta=eta)
                    bc_loss = boundary_condition_loss(E)
    
                    val_loss_history.append(helm_loss.item() + bc_loss.item())

                    k0 = feature_val_batch[:, [0]]
                    n2_max = feature_val_batch[:, [-2]]  # (B, 1)
                    n2_min = feature_val_batch[:, [-3]]   # (B, 1)
                    betas_sq = (n2_max - eta * (n2_max - n2_min)) # (B, 1)
                    betas_avg[-1] += torch.sum(betas_sq).item()
            betas_avg[-1] /= len(xyr_val)
            print('Average beta:', betas_avg[-1])

        if (iteration_no + 1) % 500 == 0 and (iteration_no + 1) in [5000, 8500, 10000] and save:
            model.to('cpu')
            checkpoint = {
                "epoch": iteration_no,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "train_loss_history": train_loss_history,
                "val_loss_history": val_loss_history,                
                "betas_avg": betas_avg,
                "xyr_val":xyr_val,
                "features_val":features_val
            }
            torch.save(checkpoint, f"checkpoint_{checkpoint_no}.pth")
            np.save('last_checkpoint.npy', checkpoint_no)
            checkpoint_no += 1
            model.to(device)
    return model, betas_avg, train_loss_history, val_loss_history

In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
device1 = 'cuda:0' if torch.cuda.is_available() else 'cpu'
device2 = 'cuda:1' if torch.cuda.is_available() and torch.cuda.device_count() > 1 else device1

In [ ]:
# torch.manual_seed(42)
dataset = RibWaveguideDataset(n_dataset=64, stochastic=True, device=device1)
loader = DataLoader(dataset, batch_size=64, shuffle=True)
print('each training epoch, a number of', len(loader.dataset), 'devices is randomly generated and used for learning')

val_dataset = RibWaveguideDataset(n_dataset=1024, stochastic=False, device=device2)
val_loader = DataLoader(val_dataset, batch_size=1024, shuffle=True)
print('there are', len(val_loader.dataset), 'samples in the validation set')

In [ ]:
model = PINO2D(in_channels=5, out_channels=2, width=64, modes1=32, modes2=32, depth=4).to(device)
model = torch.nn.DataParallel(model)

optimizer = soap.SOAP(model.parameters())
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[5000, 8500], gamma=1e-1)

In [ ]:
model, betas_avg, train_loss_history, val_loss_history = train(loader, val_loader, model, optimizer, scheduler, epochs=10001, load=False, save=True)

In [ ]:
batch_size = 4

In [ ]:
dataset = RibWaveguideDataset(device="cpu", dx=1/20, dy=1/20)

In [ ]:
devices_list = [dataset[0] for i in range(batch_size)]
xyr_list = [item[0] for item in devices_list]
features_list = [item[1] for item in devices_list]

In [ ]:
device = 'cuda'

In [ ]:
def compute_Ez_from_divergence(Ex, Ey, xy, eps, beta):
    B, _, Nx, Ny = Ex.shape

    eps_Ex = eps * Ex
    eps_Ey = eps * Ey

    x = xy[:, 0:1, :, :]
    dx_e = x[:, :, 2:, :] - x[:, :, 1:-1, :]
    dx_w = x[:, :, 1:-1, :] - x[:, :, :-2, :]

    depsEx_dx = torch.zeros_like(Ex)
    depsEx_dx[:, :, 1:-1, :] = (eps_Ex[:, :, 2:, :] - eps_Ex[:, :, :-2, :]) / (dx_e + dx_w)

    y = xy[:, 1:2, :, :]
    dy_n = y[:, :, :, 2:] - y[:, :, :, 1:-1]
    dy_s = y[:, :, :, 1:-1] - y[:, :, :, :-2]

    depsEy_dy = torch.zeros_like(Ey)
    depsEy_dy[:, :, :, 1:-1] = (eps_Ey[:, :, :, 2:] - eps_Ey[:, :, :, :-2]) / (dy_n + dy_s)

    depsEx_dx[:, :, 0, :] = depsEx_dx[:, :, 1, :]
    depsEx_dx[:, :, -1, :] = depsEx_dx[:, :, -2, :]
    depsEy_dy[:, :, :, 0] = depsEy_dy[:, :, :, 1]
    depsEy_dy[:, :, :, -1] = depsEy_dy[:, :, :, -2]

    divergence_transverse = depsEx_dx + depsEy_dy
    
    Ez = -1j * (divergence_transverse / (beta * eps))
    return Ez

In [ ]:
fundamental_colors = ["#1E90FF", "#FFFFFF", "#FF69B4"] 
custom_cmap = mcolors.LinearSegmentedColormap.from_list("black_center", fundamental_colors)

errors = []
for sample_idx, (xyr, features) in enumerate(zip(xyr_list, features_list)):

    xyr = xyr.unsqueeze(0).to(device)
    features = features.unsqueeze(0).to(device)

    k_0 = features[0, 0].item()
    core_length_x = features[0, 1].item()
    core_length_y = features[0, 2].item()
    rib_height = features[0, 3].item()

    xy = xyr[0, :2].permute(1, 2, 0)
    n_map = xyr[0, 2]

    with torch.no_grad():
        start_time = time.time()
        E_pred, beta_nn = model(xyr)
        end_time = time.time()

    print(f'FNO took {round(end_time - start_time, 5)} seconds')
    selected_component = features[:, -1].to(torch.int64)
    batch_idx = torch.arange(E_pred.shape[0], device=device)

    E = torch.zeros_like(E_pred)
    E[batch_idx, selected_component] = E_pred[:, 0]
    E[batch_idx, 1 - selected_component] = E_pred[:, 1]

    n2_max = features[:, -2]
    n2_min = features[:, -3]

    beta_sq = (n2_max - beta_nn * (n2_max - n2_min)).item()
    neff_nn = np.sqrt(beta_sq)

    with torch.no_grad():
        Ez_nn = compute_Ez_from_divergence(
            E[:, 0:1],
            E[:, 1:2],
            xyr[:, :2],
            xyr[:, 2:3],
            (beta_sq**0.5 * k_0)
        )
        # Ez_nn = compute_Ez_from_divergence(
        #     E[:, 0:1],
        #     E[:, 1:2],
        #     xyr[:, 2:3],
        #     1/20,
        #     1/20,
        #     beta_sq * k_0**2
        # )
        Ez_nn = Ez_nn.imag[0, 0].cpu()

    start_time = time.time()

    eig_vals, modes_y, modes_x, _ = full_vectorial_fd(
        xy=xy,
        eps_map=n_map,
        dx=dataset.dx,
        dy=dataset.dy,
        k0=k_0,
        num_modes=2
    )

    end_time = time.time()

    print(f'FD took {round(end_time - start_time, 5)} seconds')

    eig_vals = torch.from_numpy(eig_vals)
    modes_x = torch.from_numpy(modes_x)
    modes_y = torch.from_numpy(modes_y)

    errors.append(torch.min(torch.abs(eig_vals - neff_nn)))

    Ex_nn = E[0, 0].detach().cpu()
    Ey_nn = E[0, 1].detach().cpu()

    nn_ratio = (
        torch.linalg.norm(Ex_nn) /
        torch.linalg.norm(Ey_nn)
    ).item()

    fd_ratios = [
        (
            torch.linalg.norm(modes_x[j]) /
            torch.linalg.norm(modes_y[j])
        ).item()
        for j in range(2)
    ]

    overlaps = []

    for j in range(2):

        fdx = modes_x[j].real
        fdy = modes_y[j].real

        ov = (
            torch.abs(torch.sum(Ex_nn * fdx)) +
            torch.abs(torch.sum(Ey_nn * fdy))
        )

        overlaps.append(ov.item())
        
    overlaps_t = []
    Ex_nnt = Ey_nn.detach().clone()
    Ey_nnt = Ex_nn.detach().clone()


    for j in range(2):

        fdx = modes_x[j].real
        fdy = modes_y[j].real

        ov = (
            torch.abs(torch.sum(Ex_nnt * fdx)) +
            torch.abs(torch.sum(Ey_nnt * fdy))
        )

        overlaps_t.append(ov.item())
        
    if np.max(overlaps) > np.max(overlaps_t):
        best_fd_idx = np.argmax(overlaps)
    else:
        best_fd_idx = np.argmax(overlaps_t)
        Ex_nn = Ex_nnt
        Ey_nn = Ey_nnt
    beta_sq_fd = eig_vals[best_fd_idx].item()

    with torch.no_grad():
        Ez_fd = compute_Ez_from_divergence(
            modes_x[best_fd_idx][None, None],
            modes_y[best_fd_idx][None, None],
            xyr[:, :2].cpu(),
            xyr[:, 2:3].cpu(),
            (beta_sq_fd * k_0)
        )
        # Ez_fd = compute_Ez_from_divergence(
        #     modes_x[best_fd_idx][None, None],
        #     modes_y[best_fd_idx][None, None],
        #     xyr[:, 2:3].cpu(),
        #     1/20,
        #     1/20,
        #     (beta_sq_fd * k_0)**2
        # )
        Ez_fd = Ez_fd.imag[0, 0].cpu()
        # print('*'*10, torch.max(Ez_fd), '*'*10)

    Ex_fd = modes_x[best_fd_idx].real
    Ey_fd = modes_y[best_fd_idx].real

    dominant_is_Ex = nn_ratio > 1.0

    def normalize_field(field_x, field_y):
        max_val = max(
            torch.max(torch.abs(field_x)),
            torch.max(torch.abs(field_y))
        )

        return (
            (field_x / max_val) * torch.sgn(torch.sum(field_x)),
            (field_y / max_val) * torch.sgn(torch.sum(field_y)),
            max_val
        )

    Ex_nn, Ey_nn, max_val_nn = normalize_field(Ex_nn, Ey_nn)
    Ex_fd, Ey_fd, max_val_fd = normalize_field(Ex_fd, Ey_fd)

    Ez_nn = Ez_nn / max_val_nn * torch.sgn(torch.sum(Ez_nn))
    Ez_fd = Ez_fd / max_val_fd * torch.sgn(torch.sum(Ez_fd))

    X = xy[:, :, 0].cpu()
    Y = xy[:, :, 1].cpu()

    if dominant_is_Ex:
        dom_nn, dom_fd = Ex_nn, Ex_fd
        min_nn, min_fd = Ey_nn, Ey_fd
        dom_label = r"$E_x$"
        min_label = r"$E_y$"
        mode_label = "TE-like"
    else:
        dom_nn, dom_fd = Ey_nn, Ey_fd
        min_nn, min_fd = Ex_nn, Ex_fd
        dom_label = r"$E_y$"
        min_label = r"$E_x$"
        mode_label = "TM-like"

    def sym_clim(*fields):
        v = max(torch.max(torch.abs(f)).item() for f in fields)
        return -v, v

    row_clims = [
        sym_clim(dom_nn, dom_fd),
        sym_clim(min_nn, min_fd),
        sym_clim(Ez_nn, Ez_fd),
    ]

    def slice_ylim(*fields):
        lo = min(f.min().item() for f in fields)
        hi = max(f.max().item() for f in fields)
        pad = 0.1 * max(abs(lo), abs(hi))
        return lo - pad, hi + pad

    row_ylims = [
        slice_ylim(dom_nn, dom_fd),
        slice_ylim(min_nn, min_fd),
        slice_ylim(Ez_nn, Ez_fd),
    ]

    fig = plt.figure(figsize=(18, 8))

    wavelength = round((2 * np.pi / k_0), 3)

    fig.suptitle(
        f"{mode_label} Mode   |   λ = {wavelength} μm   |   n_eff = {neff_nn:.5f}",
        fontsize=18,
        fontweight='bold'
    )

    gs = GridSpec(
        2,
        3,
        figure=fig,
        width_ratios=[1, 1, 1],
        hspace=0.28,
        wspace=0.28
    )

    dx = dataset.dx
    dy = dataset.dy

    x_edge = int((core_length_x / 2) / dx)
    y_edge = int((core_length_y) / dy)

    center_x = X.shape[0] // 2
    center_y = X.shape[1] // 2

    edge_x = center_x + int(0.75 * x_edge)
    edge_y = center_y + int(0.75 * y_edge)

    edge_x = np.clip(edge_x, 0, X.shape[0] - 1)
    edge_y = np.clip(edge_y, 0, X.shape[1] - 1)

    if dominant_is_Ex:
        ez_slice_x = edge_x
        ez_slice_y = center_y + int(0.5 * y_edge)
    else:
        ez_slice_x = center_x
        ez_slice_y = edge_y

    slice_positions = {
        "dom": (center_x, center_y + int(0.5 * y_edge)),
        "min": (edge_x + int(0.15 * x_edge), edge_y - int(0.5 * y_edge)),
        "ez":  (ez_slice_x, ez_slice_y),
    }

    def draw_waveguide(ax):
        ax.hlines(core_length_y, -core_length_x/2, core_length_x/2, colors='black', linewidth=1.2, linestyle='-')        
        ax.hlines(0, -4.9, -core_length_x/2, colors='black', linewidth=1.2, linestyle='-')
        ax.hlines(0, core_length_x/2, 4.9, colors='black', linewidth=1.2, linestyle='-')
        ax.axhline(-rib_height, color='black', linewidth=1.5, linestyle='-')

        ax.vlines([core_length_x/2, -core_length_x/2], 0, core_length_y, colors='black', linewidth=1.2, linestyle='-')

    def draw_slice_lines(ax, slice_key):

        ix, iy = slice_positions[slice_key]

        ax.axvline(
            X[ix, iy],
            color='black',
            linestyle='--',
            linewidth=1.5,
            alpha=0.95
        )

        ax.axhline(
            Y[ix, iy],
            color='black',
            linestyle='--',
            linewidth=1.5,
            alpha=0.95
        )
    def plot_2d_heatmap(ax, Z, title, vmin, vmax, slice_key):
        norm = mcolors.SymLogNorm(linthresh=0.35 * vmax, vmin=vmin, vmax=vmax, base=10)

        ax.pcolormesh(
            X,
            Y,
            Z,
            cmap=custom_cmap,
            shading='auto',
            norm = norm
            # vmin=vmin,
            # vmax=vmax,
        )

        draw_waveguide(ax)
        draw_slice_lines(ax, slice_key)

        ax.set_title(title, fontsize=15)
        # ax.set_xlim([-3.5, 3.5])
        # ax.set_ylim([-3.5, 3.5])
        ax.tick_params(axis='x', labelsize=12) 
        ax.tick_params(axis='y', labelsize=12) 
        ax.set_xlabel('Y (μm)', fontsize=13)
        ax.set_ylabel('x (μm)', fontsize=13)
        # ax.set_xticks([])
        # ax.set_yticks([])

    def plot_1d_slices(ax, Z_nn, Z_fd, ylim, slice_key, component_name, polarization):

        ix, iy = slice_positions[slice_key]
        Z_fd = Z_fd.detach().clone()
        if torch.sum(Z_nn * Z_fd) < 0:
            Z_fd *= -1

        x_coords = X[:, iy]
        y_coords = Y[ix, :]

        ri_x = n_map[:, iy].cpu()
        ri_y = n_map[ix, :].cpu()

        ri_x = ri_x / ri_x.max()
        ri_y = ri_y / ri_y.max()

        scaled_x = ''
        scaled_y = ''

        if polarization == 'x':
            ri_y = torch.ones_like(ri_y)
            scaled_x += '$\epsilon$'    

        elif polarization == 'y':
            ri_x = torch.ones_like(ri_x)
            scaled_y += '$\epsilon$'

        elif polarization == 'z':
            ri_x = torch.ones_like(ri_x)
            ri_y = torch.ones_like(ri_y)


        ax.plot(
            x_coords[::2],
            (ri_x * Z_nn[:, iy])[::2],
            linewidth=2.5,
            color='black',
            # label= scaled_x + component_name + ' (h-slice,PINO)'
            label= scaled_x + component_name + r'$(y)$' + ' (PINO)'
        )

        ax.plot(
            x_coords[::2],
            (ri_x * Z_fd[:, iy])[::2],
            '--',
            color='red',
            linewidth=2.5,
            label= scaled_x + component_name + r'$(y)$' + ' (FD)'
            # label= scaled_x + component_name + ' (h-slice, FD)'
        )

        ax.plot(
            y_coords,
            ri_y * Z_nn[ix, :],
            linewidth=2.5,
            color='#FF69B4',
            alpha=0.9,
            # label=scaled_y + component_name + ' (v-slice,PINO)'
            label=scaled_y + component_name + r'$(x)$' +  ' (PINO)'
        )

        ax.plot(
            y_coords,
            ri_y * Z_fd[ix, :],
            '--',
            color='#1E90FF',
            linewidth=2.5,
            alpha=0.9,
            # label= scaled_y + component_name + ' (v-slice,FD)'
            label=scaled_y + component_name + r'$(x)$' +  ' (FD)'
        )

        ax.axhline(0, color='k', linestyle=':', linewidth=0.9)

        # ax.set_xlim([-3.5, 3.5])
        # ax.set_ylim(ylim)
        ax.tick_params(axis='x', labelsize=12) 

        ax.grid(alpha=0.3)

        ax.legend(frameon=False)

    heatmap_data = [
        (dom_nn, f"{dom_label}", row_clims[0], "dom"),
        (min_nn, f"{min_label}", row_clims[1], "min"),
        (Ez_nn,  r"$E_z$", row_clims[2], "ez"),
    ]

    slice_data = [
        (dom_nn, dom_fd, row_ylims[0], "dom", f"{dom_label}"),
        (min_nn, min_fd, row_ylims[1], "min", f"{min_label}"),
        (Ez_nn,  Ez_fd,  row_ylims[2], "ez",  r"$E_z$"),
    ]


    for col, (field, title, clim, slice_key) in enumerate(heatmap_data):

        ax = fig.add_subplot(gs[0, col])

        plot_2d_heatmap(
            ax,
            field,
            title,
            clim[0],
            clim[1],
            slice_key
        )

    polarizations = [
        'x' if dominant_is_Ex else 'y',
        'y' if dominant_is_Ex else 'x',
        'z'
    ]

    for col, (nn_field, fd_field, ylim, slice_key, component_name) in enumerate(slice_data):

        ax = fig.add_subplot(gs[1, col])

        plot_1d_slices(
            ax,
            nn_field,
            fd_field,
            ylim,
            slice_key,
            component_name,
            polarizations[col]
        )

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

print(
    'average error is',
    round(sum(errors).item(), 6) / len(errors)
)